# core

> Factory helpers that build a ready-to-train agent and heading arena.


In [ ]:
#| default_exp core


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
import numpy as np
from explRL.params import Params
from explRL.basis import vonMisesBasis
from explRL.agents import flexiblePolicy, PolicyGradientRL, flexibleAgent
from explRL.envs import HeadingArena


In [ ]:
#| export
def loadAgent(params=None, goal=None, seed=None):
    'Build a flexiblePolicy, PolicyGradientRL, and flexibleAgent at a random heading.'
    if params is None:
        params = Params()
    if goal is None:
        goal = np.pi
    rng = np.random.default_rng(seed)
    fk = vonMisesBasis(params.nb, params.nx, params.kappa)
    nb = params.nb
    omega = np.concatenate([np.full(nb, params.omegaF0), np.full(nb, params.omegaS0)])
    policy = flexiblePolicy(omega=omega, fk=fk, wParams=params.wParams)
    rl = PolicyGradientRL(policy, lr=params.lr)
    theta = 2 * np.pi * rng.integers(params.nx) / params.nx
    return flexibleAgent(theta, goal, policy, rl, params, rng=rng)

def loadEnv(params=None, goal=None):
    'Build a HeadingArena; goal defaults to pi radians.'
    if goal is None:
        goal = np.pi
    return HeadingArena(goal=goal)


In [ ]:
from math import pi
from fastcore.test import test_eq, test_close

p = Params()
agent = loadAgent(params=p, seed=0)
test_eq(agent.policy.omega.size, 2 * p.nb)
test_close(agent.policy.omega[:p.nb], p.omegaF0)
test_close(agent.policy.omega[p.nb:], p.omegaS0)
test_eq(agent.policy.fk.shape, (p.nb, p.nx))
test_eq(agent.rl.lr, p.lr)


In [ ]:
a0 = loadAgent(seed=42)
a1 = loadAgent(seed=42)
test_eq(a0.theta, a1.theta)
headings = {2 * pi * k / p.nx for k in range(p.nx)}
assert a0.theta in headings


In [ ]:
test_close(loadEnv().goal, pi)
test_close(loadAgent().goal, pi)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
